In [1]:
import feedparser
from newspaper import Article

# Test Each Feed

In [ ]:
def test_if_returns_full_article(feed: feedparser.FeedParserDict, print_sample: bool) -> bool:
    test_url = feed["entries"][0]["link"]
    article = Article(test_url)
    try:
        article.download()
        article.parse()
        if print_sample:
            print(article.text[:200])
        return True
    except Exception as e:
        print(f"Parsing unsuccessful. Got error: {e}")
        return False

In [3]:
def print_schema(obj, prefix="", is_last=True):
    connector = "└── " if is_last else "├── "
    
    if isinstance(obj, dict):
        items = list(obj.items())
        for i, (key, value) in enumerate(items):
            last = i == len(items) - 1
            type_name = type(value).__name__
            
            print(f"{prefix}{connector}{key}: {type_name}")
            
            extension = "    " if is_last else "│   "
            print_schema(value, prefix + extension, last)

    elif isinstance(obj, list):
        # show list summary once
        print(f"{prefix}{connector}[list]")
        
        if obj:
            extension = "    " if is_last else "│   "
            # recurse into first element only (schema representative)
            print_schema(obj[0], prefix + extension, True)

## CNBC Feed

This is the base URL for CNBC
https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=

- 10001147 – Business
- 15839135 – Earnings
- 10000664 – Finance

In [4]:
base_url = "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id="
cnbc_branch = dict(business="10001147", earnings="15839135", finance="10000664")

business_feed = feedparser.parse(base_url + cnbc_branch["business"])
earnings_feed = feedparser.parse(base_url + cnbc_branch["earnings"])
finance_feed = feedparser.parse(base_url + cnbc_branch["finance"])

cnbc_feeds = [(business_feed, "business feed"), (earnings_feed, "earnings feed"), (finance_feed, "finance feed")]

In [5]:
for feed, feed_name in cnbc_feeds:
    if test_if_returns_full_article(feed, False):
        print(f"Article for {feed_name} fetched successfully!")

Article for business feed fetched successfully!
Article for earnings feed fetched successfully!
Article for finance feed fetched successfully!


### Examine the data format of the RSS string

In [6]:
print_schema(business_feed)

└── bozo: bool
└── entries: list
    ├── [list]
    │   └── links: list
    │       ├── [list]
    │       │   └── rel: str
    │       │   └── type: str
    │       │   └── href: str
    │   └── link: str
    │   └── id: str
    │   └── guidislink: bool
    │   └── metadata_type: str
    │   └── metadata_id: str
    │   └── metadata_sponsored: str
    │   └── title: str
    │   └── title_detail: FeedParserDict
    │       ├── type: str
    │       ├── language: NoneType
    │       ├── base: str
    │       ├── value: str
    │   └── summary: str
    │   └── summary_detail: FeedParserDict
    │       ├── type: str
    │       ├── language: NoneType
    │       ├── base: str
    │       ├── value: str
    │   └── published: str
    │   └── published_parsed: struct_time
└── feed: FeedParserDict
    ├── links: list
    │   ├── [list]
    │   │   └── rel: str
    │   │   └── type: str
    │   │   └── href: str
    ├── language: str
    ├── ttl: str
    ├── title: str
    ├── title_detail:

In [7]:
business_feed["entries"][0]

{'links': [{'rel': 'alternate',
   'type': 'text/html',
   'href': 'https://www.cnbc.com/2026/04/04/china-box-office-hollywood-films.html'}],
 'link': 'https://www.cnbc.com/2026/04/04/china-box-office-hollywood-films.html',
 'id': '108286049',
 'guidislink': False,
 'metadata_type': 'cnbcnewsstory',
 'metadata_id': '108286049',
 'metadata_sponsored': 'false',
 'title': "The Chinese box office isn't the Hollywood kingmaker it used to be. Here's why",
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=10001147',
  'value': "The Chinese box office isn't the Hollywood kingmaker it used to be. Here's why"},
 'summary': 'Shifting government content controls and postpandemic Hollywood trends have made China less of a player in the global box office landscape.',
 'summary_detail': {'type': 'text/html',
  'language': None,
  'base': 'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrs

### Verify pipeline works

In [8]:
from fin_news_pipeline.ingestion.providers.cnbc import CNBCProvider
cnbc = CNBCProvider()
result = cnbc.fetch()

Unable to download body text for https://www.cnbc.com/2026/03/05/broadcom-stock-rally-ai-chip-growth.html. Error: Article `download()` failed with HTTPSConnectionPool(host='www.cnbc.com', port=443): Read timed out. (read timeout=7) on URL https://www.cnbc.com/2026/03/05/broadcom-stock-rally-ai-chip-growth.html


In [9]:
result

[RawArticle(id='CNBC_108286049', source=<Source.CNBC: 'Cnbc'>, headline="The Chinese box office isn't the Hollywood kingmaker it used to be. Here's why", summary='Shifting government content controls and postpandemic Hollywood trends have made China less of a player in the global box office landscape.', body='Hollywood has lost one of its most lucrative theatrical markets. It\'s unclear if it will ever win it back.\n\nThe Chinese box office was once a coveted space for American-made movies, so much so that studios produced films that would appeal directly to this international audience. But in the postpandemic cinema landscape, Hollywood hasn\'t generated the strong ticket sales it once saw for its biggest blockbusters — and a waning relationship with Chinese cinemas is at least partly to blame.\n\nThe U.S.-China Film Agreement, struck in 2012 between the two governments, guaranteed 34 U.S. films would be released in China each year. That pact ended in 2017 and was never renewed or ren

## Yahoo Finance

In [17]:
url = "https://feeds.finance.yahoo.com/rss/2.0/headline?s=yhoo,goog&region=US&lang=en-US"
yahoo_feed = feedparser.parse(url)

### Inspect the returned data

In [18]:
print_schema(yahoo_feed)

└── bozo: bool
└── entries: list
    ├── [list]
    │   └── summary: str
    │   └── summary_detail: FeedParserDict
    │       ├── type: str
    │       ├── language: NoneType
    │       ├── base: str
    │       ├── value: str
    │   └── id: str
    │   └── guidislink: bool
    │   └── links: list
    │       ├── [list]
    │       │   └── rel: str
    │       │   └── type: str
    │       │   └── href: str
    │   └── link: str
    │   └── published: str
    │   └── published_parsed: struct_time
    │   └── title: str
    │   └── title_detail: FeedParserDict
    │       └── type: str
    │       └── language: NoneType
    │       └── base: str
    │       └── value: str
└── feed: FeedParserDict
    ├── rights: str
    ├── rights_detail: FeedParserDict
    │   ├── type: str
    │   ├── language: NoneType
    │   ├── base: str
    │   ├── value: str
    ├── subtitle: str
    ├── subtitle_detail: FeedParserDict
    │   ├── type: str
    │   ├── language: NoneType
    │   ├── base: st

In [22]:
yahoo_feed["entries"][0]

{'summary': "Alphabet's stock has been sold off alongside the rest of the market.",
 'summary_detail': {'type': 'text/html',
  'language': None,
  'base': 'https://feeds.finance.yahoo.com/rss/2.0/headline?s=yhoo,goog&region=US&lang=en-US',
  'value': "Alphabet's stock has been sold off alongside the rest of the market."},
 'id': 'dc203556-21fe-3de0-8dfe-c8bb1eeec14d',
 'guidislink': False,
 'links': [{'rel': 'alternate',
   'type': 'text/html',
   'href': 'https://www.fool.com/investing/2026/04/04/losing-700b-market-cap-alphabet-stock/?.tsrc=rss'}],
 'link': 'https://www.fool.com/investing/2026/04/04/losing-700b-market-cap-alphabet-stock/?.tsrc=rss',
 'published': 'Sat, 04 Apr 2026 13:30:00 +0000',
 'published_parsed': time.struct_time(tm_year=2026, tm_mon=4, tm_mday=4, tm_hour=13, tm_min=30, tm_sec=0, tm_wday=5, tm_yday=94, tm_isdst=0),
 'title': 'After Losing $700 Billion in Market Cap, Is Alphabet Stock a Buy?',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': '

### Test the data

In [19]:
test_if_returns_full_article(yahoo_feed, True)

Big tech stocks have had a rough couple of months. Alphabet (GOOG 0.20%) (GOOGL 0.57%) hasn't escaped the sell-off and currently has a $3.5 trillion market cap after reaching $4.2 trillion earlier thi


True

In [24]:
from email.utils import parsedate_to_datetime
parsedate_to_datetime(yahoo_feed["entries"][0]["published"])

datetime.datetime(2026, 4, 4, 13, 30, tzinfo=datetime.timezone.utc)

### Test the pipeline

In [25]:
from fin_news_pipeline.ingestion.providers.yahoo import YahooProvider
yahoo = YahooProvider()
result = yahoo.fetch()

Unable to download body text for https://cryptonews.com/news/armstrong-coinbase-bitcoin-quantum-resistance-oversight/?.tsrc=rss. Error: Article `download()` failed with Website protected with Cloudflare, url: None on URL https://cryptonews.com/news/armstrong-coinbase-bitcoin-quantum-resistance-oversight/?.tsrc=rss
Unable to download body text for https://blockspace.media/short/comparing-quantum-computing-papers-from-google-and-caltech/?.tsrc=rss. Error: Article `download()` failed with Status code 403 for url None on URL https://blockspace.media/short/comparing-quantum-computing-papers-from-google-and-caltech/?.tsrc=rss


In [26]:
result

[RawArticle(id='Yahoo_dc203556-21fe-3de0-8dfe-c8bb1eeec14d', source=<Source.CNBC: 'Cnbc'>, headline='After Losing $700 Billion in Market Cap, Is Alphabet Stock a Buy?', summary="Alphabet's stock has been sold off alongside the rest of the market.", body="Big tech stocks have had a rough couple of months. Alphabet (GOOG 0.15%) (GOOGL 0.57%) hasn't escaped the sell-off and currently has a $3.5 trillion market cap after reaching $4.2 trillion earlier this year. That's basically the equivalent of an ExxonMobil being erased from Alphabet's market cap, so this is no small sell-off.\n\nBut is Alphabet stock worth buying now, or does it have more room to fall? Let's take a look.\n\nAlphabet recently surged in the AI arms race\n\nLast year, Alphabet was practically left for dead by investors. There were questions surrounding the future of the Google Search platform and whether Alphabet could transition into this new age of artificial intelligence (AI). However, it seamlessly made the transition